In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

# ============================================================
# 1. LOAD DATASET
# ============================================================

data_path = Path("data/raw/hotel_bookings.csv")

if not data_path.exists():
    data_path = Path("../data/raw/hotel_bookings.csv")

if not data_path.exists():
    raise FileNotFoundError(
        "Cannot find hotel_bookings.csv. Check dataset path."
    )

df = pd.read_csv(data_path)

# Target variable
target_col = "is_canceled"

if target_col not in df.columns:
    raise ValueError(
        f"Dataset không chứa biến mục tiêu '{target_col}'"
    )

print("Target validation: PASSED")

print("=" * 60)
print("1. DATASET OVERVIEW")
print("=" * 60)

print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")

print("\nFirst 5 rows:")
display(df.head())


# ============================================================
# 2. DATA TYPES AND STRUCTURE
# ============================================================

print("\n" + "=" * 60)
print("2. DATA TYPES AND STRUCTURE")
print("=" * 60)

df.info()

print("\nColumn data types:")
display(
    df.dtypes.to_frame(name="Data Type")
)

print("\nNumber of columns by data type:")
display(
    df.dtypes.value_counts().to_frame(name="Count")
)


# ============================================================
# 3. NUMERICAL AND CATEGORICAL FEATURES
# ============================================================

print("\n" + "=" * 60)
print("3. FEATURE TYPES")
print("=" * 60)

# Dựa trên kiểu dữ liệu hiện tại
numeric_cols = df.select_dtypes(
    include=np.number
).columns.tolist()

categorical_cols = df.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

# Những cột lưu dưới dạng số nhưng có ý nghĩa phân loại
semantic_categorical_cols = [
    "agent",
    "company"
]

for col in semantic_categorical_cols:
    if col in numeric_cols:
        numeric_cols.remove(col)

    if col in df.columns and col not in categorical_cols:
        categorical_cols.append(col)


# Không tính target là feature đầu vào
if target_col in numeric_cols:
    numeric_cols.remove(target_col)

if target_col in categorical_cols:
    categorical_cols.remove(target_col)

print(f"Number of numerical features: {len(numeric_cols)}")
print(f"Number of categorical features: {len(categorical_cols)}")

print("\nNumerical features:")
for col in numeric_cols:
    print("-", col)

print("\nCategorical features:")
for col in categorical_cols:
    print("-", col)

print("\nTarget variable:", target_col)


# ============================================================
# 4. MISSING VALUES ANALYSIS
# ============================================================

print("\n" + "=" * 60)
print("4. MISSING VALUES")
print("=" * 60)

missing_count = df.isna().sum()

missing_percentage = df.isna().mean() * 100

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage (%)": missing_percentage
})

missing_summary = missing_summary.sort_values(
    by="Missing Percentage (%)",
    ascending=False
)

print("\nMissing values by column:")
display(missing_summary)

print("\nColumns with missing values:")
display(
    missing_summary[
        missing_summary["Missing Count"] > 0
    ]
)

print(
    "\nTotal missing cells:",
    f"{df.isna().sum().sum():,}"
)

print(
    "Percentage of missing cells:",
    f"{df.isna().mean().mean() * 100:.2f}%"
)

print(
    "Number of columns with >= 5% missing:",
    (missing_percentage >= 5).sum()
)


# ============================================================
# 5. DUPLICATE RECORDS
# ============================================================

print("\n" + "=" * 60)
print("5. DUPLICATE RECORDS")
print("=" * 60)

duplicate_count = df.duplicated().sum()

duplicate_percentage = (
    duplicate_count / len(df) * 100
    if len(df) > 0 else 0
)

print(f"Duplicate records: {duplicate_count:,}")
print(f"Duplicate percentage: {duplicate_percentage:.2f}%")


# ============================================================
# 6. DESCRIPTIVE STATISTICS
# ============================================================

print("\n" + "=" * 60)
print("6. DESCRIPTIVE STATISTICS")
print("=" * 60)

print("\nNumerical features:")
display(
    df[numeric_cols].describe().T
)

print("\nCategorical features:")
display(
    df[categorical_cols].astype("object").describe().T
)


# ============================================================
# 7. TARGET DISTRIBUTION
# ============================================================

print("\n" + "=" * 60)
print("7. TARGET DISTRIBUTION")
print("=" * 60)

if target_col in df.columns:

    target_summary = pd.DataFrame({
        "Count": df[target_col].value_counts(),
        "Percentage (%)":
            df[target_col].value_counts(normalize=True) * 100
    })

    display(target_summary)


# ============================================================
# 8. CHECK DATASET REQUIREMENTS
# ============================================================

print("\n" + "=" * 60)
print("8. DATASET REQUIREMENTS CHECK")
print("=" * 60)

requirements = {
    "At least 10,000 records":
        df.shape[0] >= 10000,

    "At least 10 features":
        df.shape[1] - 1 >= 10,

    "Contains numerical features":
        len(numeric_cols) > 0,

    "Contains categorical features":
        len(categorical_cols) > 0,

    "At least one feature has >= 5% missing":
        (df.drop(columns=[target_col], errors="ignore")
           .isna().mean() >= 0.05).any()
}

requirements_df = pd.DataFrame({
    "Requirement": list(requirements.keys()),
    "Passed": list(requirements.values())
})

requirements_df["Status"] = requirements_df["Passed"].map({
    True: "PASS",
    False: "FAIL"
})

display(requirements_df[["Requirement", "Status"]])

if all(requirements.values()):
    print("\nDataset satisfies all basic requirements!")
else:
    print("\nDataset does not satisfy all basic requirements.")


# ============================================================
# 9. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("9. FINAL DATASET SUMMARY")
print("=" * 60)

summary = pd.DataFrame({
    "Metric": [
        "Number of records",
        "Number of columns",
        "Numerical features",
        "Categorical features",
        "Missing cells",
        "Columns with >= 5% missing",
        "Duplicate records",
        "Target variable"
    ],
    "Value": [
        df.shape[0],
        df.shape[1],
        len(numeric_cols),
        len(categorical_cols),
        int(df.isna().sum().sum()),
        int((missing_percentage >= 5).sum()),
        int(duplicate_count),
        target_col
    ]
})

display(summary)

Target validation: PASSED
1. DATASET OVERVIEW
Number of rows: 119,390
Number of columns: 32

First 5 rows:


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03



2. DATA TYPES AND STRUCTURE
<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12

,Data Type
hotel,str
is_canceled,int64
lead_time,int64
arrival_date_year,int64
arrival_date_month,str
arrival_date_week_number,int64
arrival_date_day_of_month,int64
stays_in_weekend_nights,int64
stays_in_week_nights,int64
adults,int64



Number of columns by data type:


...

,Count,Percentage (%)
is_canceled,,
0,75166,62.958372
1,44224,37.041628



8. DATASET REQUIREMENTS CHECK


,Requirement,Status
0,"At least 10,000 records",PASS
1,At least 10 features,PASS
2,Contains numerical features,PASS
3,Contains categorical features,PASS
4,At least one feature has >= 5% missing,PASS



Dataset satisfies all basic requirements!

9. FINAL DATASET SUMMARY


,Metric,Value
0,Number of records,119390
1,Number of columns,32
2,Numerical features,17
3,Categorical features,14
4,Missing cells,129425
5,Columns with >= 5% missing,2
6,Duplicate records,31994
7,Target variable,is_canceled
